# Notebook 02 — Colector Interactivo de Puntos de Control (5 Clases + Capa Guía de Mangle)

- **Instancia SEPAL recomendada:** `t2` o `m2` (0.04–0.10 USD/h).
- **Propósito:**
  1. Cargar el **Stack Biestacional** y la **capa guía actual de manglar** (`projects/tesismangle/assets/DinMangle_2020_2022_V23`).
  2. Permitirte **dibujar y guardar puntos de control directamente sobre el mapa** dentro de Jupyter para las **5 clases del protocolo**:
     - `1`: Mangle
     - `2`: Bosque latifoliado
     - `3`: Agua
     - `4`: Infraestructura urbana
     - `5`: Pastos/Ganadería
  3. *(Opcional)* Generar una propuesta inicial de puntos dentro del núcleo erosionado (20 m) de la capa guía de manglar para agilizar la captura de la clase `1`.
  4. Limpiar duplicados dentro de la misma celda de **10 × 10 m** (evita fuga espacial), verificar que caigan dentro de `MASCARA_COMUN == 1` y generar el split preliminar **70% Entrenamiento / 30% Prueba Preliminar**.

> **Nota metodológica:** El split 70/30 sirve para el entrenamiento de Random Forest y el diagnóstico preliminar antes de exportar los mapas. La evaluación oficial de precisión y área se realizará después en **AREA2 + Collect Earth Online (Notebook 05)**.

In [ ]:
# 1. Configuracion y carga de insumos
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import ee
import geemap
import geopandas as gpd
import ipywidgets as widgets
import numpy as np
import pandas as pd
from IPython.display import clear_output, display
from shapely.geometry import Point, shape
from sklearn.model_selection import train_test_split

from src.config_utils import (
    check_asset_exists,
    ensure_output_dirs,
    get_class_maps,
    init_gee,
    load_aoi_geometry,
    load_config,
)
from src.stack_builder import build_biseasonal_stack, get_stack_asset_ids

# >>> CAMBIAR AQUI ENTRE "PACIFICO" Y "CARIBE" <<<
REGION = "PACIFICO"
USAR_STACK_EXPORTADO = True  # False si aun no ha terminado el Export del NB 01

cfg = load_config()
dirs = ensure_output_dirs(cfg)
init_gee(cfg)

CLASES_NOMBRES, CLASES_CORTOS, CLASES_COLORES = get_class_maps(cfg)
aoi_base = load_aoi_geometry(cfg["assets"]["aoi"][REGION])
stack_ids = get_stack_asset_ids(REGION, cfg)

if USAR_STACK_EXPORTADO and check_asset_exists(stack_ids["stack"])["existe"]:
    stack_img = ee.Image(stack_ids["stack"])
    qa_img = ee.Image(stack_ids["qa"])
    print("Usando Stack exportado:", stack_ids["stack"])
else:
    print("Stack exportado no disponible aun; construyendo vista al vuelo para el mapa...")
    b = build_biseasonal_stack(REGION, cfg)
    stack_img = b["stack"]
    qa_img = b["qa"]

# Archivo persistente de avance (EPSG:4326) en disco de SEPAL
GPKG_RAW = dirs["muestras"] / f"puntos_control_brutos_{REGION.lower()}.gpkg"
print("Archivo persistente de colecta:", GPKG_RAW)

In [ ]:
# 2. Inspeccionar la capa guia actual de manglar (DinMangle_2020_2022_V23)
guia_id = cfg["assets"]["mangle_guia"]
info_guia = check_asset_exists(guia_id)
print("Info capa guia:", info_guia)

if info_guia["existe"] and info_guia["tipo"] == "IMAGE":
    guia_img = ee.Image(guia_id)
    banda_guia = info_guia["bandas"][0]
    hist = guia_img.select(banda_guia).reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=aoi_base,
        scale=30,
        bestEffort=True,
        maxPixels=int(cfg["gee"]["max_pixels"]),
    ).get(banda_guia).getInfo()
    print(f"Valores presentes en '{banda_guia}' dentro del AOI {REGION}:", hist)
elif info_guia["existe"] and info_guia["tipo"] == "TABLE":
    guia_fc = ee.FeatureCollection(guia_id).filterBounds(aoi_base)
    print(f"Poligonos en capa guia dentro de {REGION}:", guia_fc.size().getInfo())
    print("Propiedades:", info_guia.get("propiedades", []))

In [ ]:
# 3. (Opcional) Generar puntos candidatos iniciales de Mangle (Clase 1) desde DinMangle
# Util si quieres pre-poblar una base de puntos de Mangle en el interior de la capa guia
# y luego revisarlos/completar las otras 4 clases en el mapa interactivo.

GENERAR_CANDIDATOS_MANGLE = False
VALOR_CLASE_MANGLE_EN_GUIA = 1  # Ajustar segun el histograma de la celda anterior si es IMAGE
N_CANDIDATOS_MANGLE = 120       # Numero de puntos guia a proponer

def cargar_puntos_existentes(path: Path) -> gpd.GeoDataFrame:
    if path.exists():
        gdf = gpd.read_file(path)
        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:4326")
        return gdf.to_crs("EPSG:4326")
    return gpd.GeoDataFrame(
        columns=["clase_id", "clase", "fuente", "region", "geometry"],
        geometry="geometry",
        crs="EPSG:4326",
    )

gdf_puntos = cargar_puntos_existentes(GPKG_RAW)

if GENERAR_CANDIDATOS_MANGLE:
    erosion_m = float(cfg["muestreo"]["erosion_mangle_guia_m"])
    if info_guia["tipo"] == "IMAGE":
        banda_guia = info_guia["bandas"][0]
        mask_mangle = (
            ee.Image(guia_id)
            .select(banda_guia)
            .eq(VALOR_CLASE_MANGLE_EN_GUIA)
            .selfMask()
            .focalMin(radius=erosion_m, units="meters")
            .updateMask(qa_img.select("MASCARA_COMUN").eq(1))
            .updateMask(qa_img.select("MASCARA_AOI_BASE").eq(1))
        )
        pts_fc = mask_mangle.stratifiedSample(
            numPoints=N_CANDIDATOS_MANGLE,
            classBand=banda_guia,
            region=aoi_base,
            scale=10,
            seed=int(cfg["proyecto"]["seed"]),
            geometries=True,
        )
    else:
        geom_mangle = ee.FeatureCollection(guia_id).filterBounds(aoi_base).geometry().buffer(-erosion_m, 1)
        pts_fc = ee.FeatureCollection.randomPoints(
            region=geom_mangle, points=N_CANDIDATOS_MANGLE, seed=int(cfg["proyecto"]["seed"])
        )

    feats = pts_fc.getInfo()["features"]
    nuevos = gpd.GeoDataFrame(
        [
            {
                "clase_id": 1,
                "clase": CLASES_NOMBRES[1],
                "fuente": "guia_DinMangle",
                "region": REGION,
                "geometry": shape(f["geometry"]),
            }
            for f in feats
        ],
        geometry="geometry",
        crs="EPSG:4326",
    )
    gdf_puntos = pd.concat([gdf_puntos, nuevos], ignore_index=True)
    gdf_puntos = gpd.GeoDataFrame(gdf_puntos, geometry="geometry", crs="EPSG:4326")
    gdf_puntos.to_file(GPKG_RAW, driver="GPKG")
    print(f"Se agregaron {len(nuevos)} candidatos de Mangle. Total guardado: {len(gdf_puntos)}")
else:
    print(f"Puntos actualmente guardados en {GPKG_RAW.name}: {len(gdf_puntos)}")
    if len(gdf_puntos) > 0:
        display(gdf_puntos.groupby(["clase_id", "clase"]).size().rename("n_puntos").to_frame())

In [ ]:
# 4. COLECTOR INTERACTIVO DE PUNTOS DE CONTROL EN JUPYTER (geemap + ipywidgets)
# Instrucciones:
#   1) Selecciona la clase en el desplegable.
#   2) Usa la herramienta de marcador (pin) o poligono en la barra izquierda del mapa.
#   3) Haz clic en 'Guardar puntos dibujados' (se persiste al instante en el .gpkg).
#   4) Alterna las capas (Satelite, Falsocolor B8-B11-B4, SAR VH y DinMangle) segun necesites.

m = geemap.Map(height="620px")
m.centerObject(aoi_base, 11)
m.add_basemap("SATELLITE")

m.addLayer(stack_img, {"bands": ["B8_SECA", "B11_SECA", "B4_SECA"], "min": 0, "max": 0.40}, "Falsocolor Manglar (B8-B11-B4 Seca)", True)
m.addLayer(stack_img, {"bands": ["B4_SECA", "B3_SECA", "B2_SECA"], "min": 0, "max": 0.30}, "RGB Seca (S2)", False)
m.addLayer(stack_img, {"bands": ["B4_LLUVIA", "B3_LLUVIA", "B2_LLUVIA"], "min": 0, "max": 0.30}, "RGB Lluvia (S2)", False)
m.addLayer(stack_img.select("VH_SECA"), {"min": -28, "max": -8, "palette": ["0015FF", "00FFFF", "FFFF00", "FF0000"]}, "SAR VH Seca (dB)", False)
m.addLayer(aoi_base, {"color": "FFFFFF"}, f"AOI {REGION}", True)

if info_guia["existe"]:
    if info_guia["tipo"] == "IMAGE":
        m.addLayer(
            ee.Image(guia_id).select(0).selfMask(),
            {"min": 1, "max": 5, "palette": ["00FF66", "FFFF00", "FF3333", "00AAFF", "AA00FF"]},
            "Guia DinMangle 2020-2022",
            True,
            0.55,
        )
    else:
        m.addLayer(ee.FeatureCollection(guia_id).filterBounds(aoi_base), {"color": "00FF66"}, "Guia DinMangle 2020-2022", True)

# Mostrar puntos ya guardados por clase
def _dibujar_puntos_guardados():
    gdf_curr = cargar_puntos_existentes(GPKG_RAW)
    if len(gdf_curr) == 0:
        return
    for cid, grp in gdf_curr.groupby("clase_id"):
        cid_int = int(cid)
        fc_cls = geemap.gdf_to_ee(grp[["clase_id", "clase", "geometry"]])
        m.addLayer(fc_cls, {"color": CLASES_COLORES[cid_int]}, f"Puntos {cid_int}: {CLASES_NOMBRES[cid_int]} ({len(grp)})")

_dibujar_puntos_guardados()

# Controles interactivos
dd_clase = widgets.Dropdown(
    options=[(f"{cid} - {nom}", cid) for cid, nom in CLASES_NOMBRES.items()],
    value=1,
    description="Clase:",
    style={"description_width": "initial"},
)
btn_guardar = widgets.Button(description="Guardar puntos dibujados", button_style="success", icon="save")
btn_deshacer = widgets.Button(description="Deshacer ultimo lote", button_style="warning", icon="undo")
out_log = widgets.Output()

HISTORIAL_LOTES = []

def _mostrar_resumen(gdf):
    if len(gdf) == 0:
        print("Aun no hay puntos guardados.")
        return
    res = gdf.groupby(["clase_id", "clase"]).size().rename("n_puntos").reset_index()
    display(res)
    print(f"Total acumulado: {len(gdf)} puntos (archivo: {GPKG_RAW.name})")

def on_guardar_click(_):
    with out_log:
        clear_output(wait=True)
        # Priorizar draw_control.data (GeoJSON local instantaneo) con respaldo en draw_features
        features = list(getattr(m.draw_control, "data", None) or [])
        if not features:
            features = list(getattr(m, "draw_features", None) or [])
        if not features:
            print("No hay geometrias dibujadas en el mapa. Dibuja al menos un marcador o poligono primero.")
            return
        cid = int(dd_clase.value)
        nuevos_rows = []
        for feat in features:
            # Soportar tanto dict GeoJSON como ee.Feature
            if isinstance(feat, dict):
                geom_obj = shape(feat["geometry"])
            else:
                geom_obj = shape(feat.geometry().getInfo())
            if geom_obj.geom_type == "Point":
                pt = geom_obj
            elif geom_obj.geom_type == "MultiPoint":
                for p in geom_obj.geoms:
                    nuevos_rows.append({"clase_id": cid, "clase": CLASES_NOMBRES[cid], "fuente": "dibujo_manual_sepal", "region": REGION, "geometry": p})
                continue
            else:
                # Si dibujas un poligono, se toma su punto representativo interior
                pt = geom_obj.representative_point()
            nuevos_rows.append({"clase_id": cid, "clase": CLASES_NOMBRES[cid], "fuente": "dibujo_manual_sepal", "region": REGION, "geometry": pt})

        gdf_curr = cargar_puntos_existentes(GPKG_RAW)
        gdf_new = gpd.GeoDataFrame(nuevos_rows, geometry="geometry", crs="EPSG:4326")
        gdf_all = gpd.GeoDataFrame(pd.concat([gdf_curr, gdf_new], ignore_index=True), geometry="geometry", crs="EPSG:4326")
        gdf_all.to_file(GPKG_RAW, driver="GPKG")
        HISTORIAL_LOTES.append(len(gdf_new))

        # Limpiar dibujos temporales del mapa y vaciar el cache interno de geemap
        m.draw_control.clear()
        if hasattr(m, "draw_features"):
            m.draw_features = []
        if hasattr(m, "user_rois"):
            m.user_rois = None
        # Pintar visualmente el lote recien guardado en el mapa
        try:
            m.add_gdf(gdf_new[["clase_id", "clase", "geometry"]], layer_name=f"Lote {len(HISTORIAL_LOTES)}: {CLASES_NOMBRES[cid]} ({len(gdf_new)})")
        except Exception:
            pass
        print(f"[OK] Guardados {len(gdf_new)} punto(s) en Clase {cid} ({CLASES_NOMBRES[cid]}).")
        _mostrar_resumen(gdf_all)

def on_deshacer_click(_):
    with out_log:
        clear_output(wait=True)
        if not HISTORIAL_LOTES:
            print("No hay lotes recientes en esta sesion para deshacer.")
            return
        n_rem = HISTORIAL_LOTES.pop()
        gdf_curr = cargar_puntos_existentes(GPKG_RAW)
        gdf_curr = gdf_curr.iloc[:-n_rem].copy()
        gdf_curr.to_file(GPKG_RAW, driver="GPKG")
        print(f"Se eliminaron los ultimos {n_rem} punto(s).")
        _mostrar_resumen(gdf_curr)

btn_guardar.on_click(on_guardar_click)
btn_deshacer.on_click(on_deshacer_click)

display(widgets.HBox([dd_clase, btn_guardar, btn_deshacer]))
display(out_log)
display(m)

In [ ]:
# 5. Limpieza espacial (deduplicacion a celda exacta de 10 m) y Split preliminar 70/30
gdf_raw = cargar_puntos_existentes(GPKG_RAW)
assert len(gdf_raw) > 0, "Primero colecta puntos de control en la celda anterior."

crs_utm = cfg["regiones"][REGION]["crs"]
gdf_utm = gdf_raw.to_crs(crs_utm).copy()

# Deduplicar puntos que caigan en el mismo pixel de 10 x 10 m (evita fuga espacial train/test)
gdf_utm["px_col"] = np.floor(gdf_utm.geometry.x / 10.0).astype(int)
gdf_utm["px_row"] = np.floor(gdf_utm.geometry.y / 10.0).astype(int)
n_antes = len(gdf_utm)
gdf_clean = gdf_utm.drop_duplicates(subset=["px_col", "px_row"], keep="first").copy()
n_duplicados = n_antes - len(gdf_clean)

# Volver a EPSG:4326 y agregar lon/lat explicitos
gdf_clean = gdf_clean.to_crs("EPSG:4326")
gdf_clean["lon"] = gdf_clean.geometry.x
gdf_clean["lat"] = gdf_clean.geometry.y
gdf_clean["sample_id"] = np.arange(1, len(gdf_clean) + 1)

# Verificar que las 5 clases tengan suficientes muestras para split estratificado 70/30
conteos = gdf_clean["clase_id"].value_counts().sort_index()
print(f"Puntos brutos: {n_antes} | Duplicados en mismo pixel 10m eliminados: {n_duplicados} | Limpios: {len(gdf_clean)}")
display(conteos.rename("n_limpios").to_frame())

if (conteos >= 4).all() and len(conteos) >= 2:
    idx_train, idx_test = train_test_split(
        gdf_clean.index,
        test_size=float(cfg["muestreo"]["test_size_preliminar"]),
        stratify=gdf_clean["clase_id"],
        random_state=int(cfg["proyecto"]["seed"]),
    )
    gdf_clean["split"] = "train"
    gdf_clean.loc[idx_test, "split"] = "test_preliminar"

    display(pd.crosstab(gdf_clean["clase"], gdf_clean["split"], margins=True))

    out_gpkg = dirs["muestras"] / f"muestras_limpias_{REGION.lower()}_70_30.gpkg"
    out_csv = dirs["muestras"] / f"muestras_limpias_{REGION.lower()}_70_30.csv"
    gdf_clean.to_file(out_gpkg, driver="GPKG")
    gdf_clean.drop(columns="geometry").to_csv(out_csv, index=False)
    print(f"[OK] Guardado: {out_gpkg}")
    print(f"[OK] Guardado: {out_csv}")
else:
    print("[AVISO] Asegurate de tener al menos 4 puntos por clase en las 5 clases antes del split 70/30.")

In [ ]:
# 6. Subir las muestras limpias (70/30) como Asset permanente en GEE
SUBIR_MUESTRAS_A_GEE = False

if SUBIR_MUESTRAS_A_GEE:
    out_gpkg = dirs["muestras"] / f"muestras_limpias_{REGION.lower()}_70_30.gpkg"
    gdf_export = gpd.read_file(out_gpkg)
    fc_muestras = geemap.gdf_to_ee(gdf_export[["sample_id", "clase_id", "clase", "split", "fuente", "region", "geometry"]])
    ver = cfg["proyecto"]["version_pipeline"].lower()
    asset_dest = cfg["gee"]["asset_root"].rstrip("/") + f"/muestras_control_{REGION.lower()}_{ver}"

    task = ee.batch.Export.table.toAsset(
        collection=fc_muestras,
        description=f"Asset_Muestras_Control_{REGION}_{ver}",
        assetId=asset_dest,
    )
    task.start()
    print(f"Exportacion iniciada -> {asset_dest} (task_id: {task.id})")
else:
    print("Cambia SUBIR_MUESTRAS_A_GEE = True cuando tus 5 clases esten completas y balanceadas.")